In [ ]:
!pip install spikingjelly -q
!pip install decord -q
!pip install brian2 -q
import brian2
import os
import math
from sklearn.manifold import TSNE
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np
import torchvision.transforms as transforms
from PIL import Image
from spikingjelly.activation_based import base, layer, neuron, surrogate, functional
import pandas as pd
from pathlib import Path
import random
from torch.utils.data import Dataset, DataLoader
from decord import VideoReader, cpu
import decord
from random import choice
from tqdm import tqdm
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, f1_score, precision_score,recall_score 
import itertools
from brian2 import *
import seaborn as sns
import json
import cv2
import optuna
import torch.nn.functional as F
device = torch.device(
        'cuda' if torch.cuda.is_available() else 'cpu'
    )

json_path = '/kaggle/input/datasets/mohammadmehranfar/wang-triplet-input-train/wang_inputs_train.json'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 437.6/437.6 kB 7.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 85.1 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is inco

In [ ]:
def train_lora_feature_extractor(model, train_loader, optimizer, criterion, epochs=3, save_dir="./lora_checkpoints"):
    model.to(device)
    model.train()
    os.makedirs(save_dir, exist_ok=True)
    best_loss = float('inf')

    scaler = torch.amp.GradScaler('cuda')

    print("\nStarting Fast LoRA Fine-Tuning Phase...")
    print(f"Checkpoints will be saved in: {save_dir}\n")

    for epoch in range(epochs):
        running_loss = 0.0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")

        for batch in pbar:
            v_a = batch['anchor'].to(device)
            v_p = batch['positive'].to(device)
            v_n = batch['negative'].to(device)

            v_a = v_a.permute(2, 0, 1, 3, 4).contiguous()
            v_p = v_p.permute(2, 0, 1, 3, 4).contiguous()
            v_n = v_n.permute(2, 0, 1, 3, 4).contiguous()

            optimizer.zero_grad()

            with torch.amp.autocast('cuda'):
                # Pass Anchor
                functional.reset_net(model)
                out_a = model(v_a)
                z_a = out_a.mean(dim=[2, 3, 4]) if out_a.dim(
                ) == 5 else out_a.flatten(start_dim=1)
                z_a = F.normalize(z_a, p=2, dim=-1)

                # Pass Positive
                functional.reset_net(model)
                out_p = model(v_p)
                z_p = out_p.mean(dim=[2, 3, 4]) if out_p.dim(
                ) == 5 else out_p.flatten(start_dim=1)
                z_p = F.normalize(z_p, p=2, dim=-1)

                # Pass Negative
                functional.reset_net(model)
                out_n = model(v_n)
                z_n = out_n.mean(dim=[2, 3, 4]) if out_n.dim(
                ) == 5 else out_n.flatten(start_dim=1)
                z_n = F.normalize(z_n, p=2, dim=-1)

                loss = criterion(z_a, z_p, z_n)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item()
            pbar.set_postfix({'loss': f"{loss.item():.4f}"})

            del v_a, v_p, v_n, out_a, out_p, out_n, z_a, z_p, z_n, loss

        epoch_loss = running_loss / len(train_loader)
        print(f"\nEpoch [{epoch+1}/{epochs}] - Triplet Loss: {epoch_loss:.4f}")

        checkpoint_path = os.path.join(
            save_dir, f"lora_checkpoint_epoch_{epoch+1}.pth")
        torch.save(model.state_dict(), checkpoint_path)

        if epoch_loss < best_loss:
            best_loss = epoch_loss
            best_model_path = os.path.join(save_dir, "lora_best_model.pth")
            torch.save(model.state_dict(), best_model_path)
            print(f"New Best Model Saved (Loss: {best_loss:.4f})")

        print("-" * 50)

    print("LoRA Training Completed!")
    return model


def extract_and_save_wang_inputs(model, dataloader, save_json_path, base_rate=15.0, scale=30.0):
    model.to(device)
    model.eval()

    results = []
    print(f"\nExtracting Features & Firing Rates -> {save_json_path}")

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Extracting Features"):
            v_a = batch['anchor'].to(device)
            v_p = batch['positive'].to(device)
            v_n = batch['negative'].to(device)

            v_a = v_a.permute(2, 0, 1, 3, 4).contiguous()
            v_p = v_p.permute(2, 0, 1, 3, 4).contiguous()
            v_n = v_n.permute(2, 0, 1, 3, 4).contiguous()

            with torch.amp.autocast('cuda'):
                # Anchor
                functional.reset_net(model)
                out_a = model(v_a)
                z_a = out_a.mean(dim=[2, 3, 4]) if out_a.dim(
                ) == 5 else out_a.flatten(start_dim=1)
                z_a = F.normalize(z_a, p=2, dim=-1)

                # Positive
                functional.reset_net(model)
                out_p = model(v_p)
                z_p = out_p.mean(dim=[2, 3, 4]) if out_p.dim(
                ) == 5 else out_p.flatten(start_dim=1)
                z_p = F.normalize(z_p, p=2, dim=-1)

                # Negative
                functional.reset_net(model)
                out_n = model(v_n)
                z_n = out_n.mean(dim=[2, 3, 4]) if out_n.dim(
                ) == 5 else out_n.flatten(start_dim=1)
                z_n = F.normalize(z_n, p=2, dim=-1)

                dist_pos = F.pairwise_distance(z_a, z_p)
                dist_neg = F.pairwise_distance(z_a, z_n)

                rate_pos = base_rate + scale * dist_pos
                rate_neg = base_rate + scale * dist_neg

            batch_size = v_a.shape[1] if v_a.dim() == 5 else v_a.shape[0]

            for b in range(batch_size):
                d_p_val = float(dist_pos[b].cpu().item()) if dist_pos.dim(
                ) > 0 else float(dist_pos.cpu().item())
                d_n_val = float(dist_neg[b].cpu().item()) if dist_neg.dim(
                ) > 0 else float(dist_neg.cpu().item())
                r_p_val = float(rate_pos[b].cpu().item()) if rate_pos.dim(
                ) > 0 else float(rate_pos.cpu().item())
                r_n_val = float(rate_neg[b].cpu().item()) if rate_neg.dim(
                ) > 0 else float(rate_neg.cpu().item())

                t_id = batch['triplet_id'][b] if isinstance(
                    batch['triplet_id'], list) else batch['triplet_id']
                t_id_val = t_id.item() if torch.is_tensor(t_id) else t_id

                triplet_info = {
                    'triplet_id': int(t_id_val),
                    'anchor': batch['video_names'][0][b],
                    'positive': batch['video_names'][1][b],
                    'negative': batch['video_names'][2][b],
                    'dist_pos': d_p_val,
                    'dist_neg': d_n_val,
                    'rate_pos_hz': r_p_val,
                    'rate_neg_hz': r_n_val
                }
                results.append(triplet_info)

    with open(save_json_path, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=4)

    print(f"Saved Firing Rates to: {save_json_path}")
    return results


save_path = '/kaggle/input/models/mohammadmehranfar/lorafbsnet-weights/pytorch/default/1/best_model.pth'
full_model = load_model_from_state_dict(save_path, num_classes=3)
extractor_model = LoRaFBSNetFeatureExtractor(full_model)
set_step_mode(extractor_model, 'm', (ConvRecurrentContainer,))

trainable_params = [p for p in extractor_model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=5e-4, weight_decay=1e-2)
criterion = nn.TripletMarginLoss(margin=0.3, p=2)

trained_extractor = train_lora_feature_extractor(
    model=extractor_model,
    train_loader=train_loader,
    optimizer=optimizer,
    criterion=criterion,
    epochs=3,
)

checkpoint_path = "lora_trained_weights.pth"
torch.save(trained_extractor.state_dict(), checkpoint_path)


train_rates = extract_and_save_wang_inputs(
    model=trained_extractor,
    dataloader=test_loader, 
    save_json_path="wang_inputs_train.json",
)

In [ ]:
defaultclock.dt = 0.5 * ms
dt_sim = defaultclock.dt

# Population proportions
f_E = 0.15
f_inh = 0.2
N_tot = 2000

NE = int(N_tot * (1.0 - f_inh))
NI = int(N_tot * f_inh)
N_D = int(f_E * NE)

# Synaptic weight scalings
w_p = 1.6
w_m = (1.0 - f_E * (w_p - 1.0) / (1.0 - f_E)) * 0.9

# Synaptic conductances
gEE_AMPA = 0.05 * nS
gEE_NMDA = 0.165 * nS
gEI_AMPA = 0.04 * nS
gEI_NMDA = 0.13 * nS
gIE_GABA = 1.6 * nS
gII_GABA = 1.1 * nS

d_rec = 0.5 * ms

# External input setup
gextE = 2.4 * nS
gextI = 1.62 * nS
nu_ext_sel = 2420 * Hz
nu_ext_nsel = 2400 * Hz
w_ext_stim_val = 0.17

# Biophysical properties
CmE = 0.5 * nF
CmI = 0.2 * nF
gLeakE = 25.0 * nS
gLeakI = 20.0 * nS
Vl = -70.0 * mV
Vt = -60.0 * mV
Vr = -55.0 * mV
tau_refE = 1.0 * ms
tau_refI = 1.0 * ms

VrevE = 0 * mV
VrevI = -70 * mV
tau_AMPA = 2.0 * ms
tau_GABA = 5.0 * ms
tau_NMDA_decay = 100.0 * ms
tau_NMDA_rise = 2.0 * ms
alpha_NMDA = 0.5 * kHz

# Protocol timing
stim_on = 50.0 * ms
stim_off = 2000.0 * ms
trial_runtime = 3000.0 * ms

# Decision threshold criteria
win_thr_spikes = 5
margin_thr = 2
rt_bin_ms = 5.0
rt_bin = rt_bin_ms * ms

# ------------------------------------------------------------------------------
# 2. Network Circuit Construction
# ------------------------------------------------------------------------------


def make_wang_circuit(num_classes):
    eqsE = '''
    dV/dt = (-gea*(V-VrevE) - gen*(V-VrevE)/(1.0+exp(-V/mV*0.062)/3.57) - gi*(V-VrevI) - (V-Vl)) / (tau): volt
    dgea/dt = -gea/(tau_AMPA) : 1
    dgi/dt = -gi/(tau_GABA) : 1
    dspre/dt = -spre/(tau_NMDA_decay)+alpha_NMDA*xpre*(1-spre) : 1
    dxpre/dt= -xpre/(tau_NMDA_rise) : 1
    gen : 1
    tau : second
    '''

    eqsI = '''
    dV/dt = (-gea*(V-VrevE) - gen*(V-VrevE)/(1.0+exp(-V/mV*0.062)/3.57) - gi*(V-VrevI) - (V-Vl)) / (tau): volt
    dgea/dt = -gea/(tau_AMPA) : 1
    dgi/dt = -gi/(tau_GABA) : 1
    gen : 1
    tau : second
    '''

    decisionE = NeuronGroup(
        NE, model=eqsE,
        threshold='V > Vt', reset='V = Vr',
        refractory=tau_refE, method='euler'
    )
    decisionI = NeuronGroup(
        NI, model=eqsI,
        threshold='V > Vt', reset='V = Vr',
        refractory=tau_refI, method='euler'
    )
    decisionE.tau = CmE / gLeakE
    decisionI.tau = CmI / gLeakI

    DE_list = []
    start_idx = 0
    for _ in range(num_classes):
        DE_list.append(decisionE[start_idx:start_idx + N_D])
        start_idx += N_D
    # Remaining Neurons are non selective Known as decisionNE
    decisionEN = decisionE[start_idx:]

    ampa_blocks = []

    # Preparing Synapses (AMPA)

    def make_ampa_block(src, tgt, w):
        # When preSynaptic Neuron spikes w would update to w+gea with d_rec delay from pre to post synaptic
        syn = Synapses(src, tgt, 'w: 1', on_pre='gea += w', delay=d_rec)

        '''Synapses(
           src:  A group of neurons that send Spikes (pre Synaptic)
           tgt: A group of neurons that recieves spikes (post synaptic)
           model: Variables and Synapse equations
           on_pre: Operation that we want to happen when spikes arrives
           dely: delay between spike and the effect on the target
        )
        '''

        # All to All connection
        syn.connect()
        syn.w = w
        ampa_blocks.append(syn)

    wA_within = w_p * gEE_AMPA / gLeakE
    wA_cross = w_m * gEE_AMPA / gLeakE
    wA_dn = gEE_AMPA / gLeakE

    # Within class AMPA Synapases
    for k in range(num_classes):
        make_ampa_block(DE_list[k], DE_list[k], wA_within)

    # Between class AMPA Synapses
    for i_cls in range(num_classes):
        for j_cls in range(num_classes):
            if i_cls != j_cls:
                make_ampa_block(DE_list[i_cls], DE_list[j_cls], wA_cross)

    # Non selective Neurons AMPA Synapses
    for k in range(num_classes):
        # class -> DN
        make_ampa_block(DE_list[k], decisionEN, wA_dn)
        # DN -> class
        make_ampa_block(decisionEN, DE_list[k], wA_cross)
    # DN -> DN
    make_ampa_block(decisionEN, decisionEN, wA_dn)

    C_EI_A = Synapses(decisionE, decisionI, 'w: 1',
                      on_pre='gea += w', delay=d_rec)
    C_EI_A.connect()
    C_EI_A.w = gEI_AMPA / gLeakI

    # Self excitation for synaptic updating NMDA
    selfnmda = Synapses(decisionE, decisionE, 'w:1',
                        on_pre='xpre_post += w', delay=d_rec)
    selfnmda.connect(j='i')
    selfnmda.w = 1

    @network_operation()
    def update_nmda():
        s_sel = [np.sum(np.asarray(pop.spre)) for pop in DE_list]
        s_dn = np.sum(np.asarray(decisionEN.spre))
        s_all = np.sum(s_sel) + s_dn

        for k, pop in enumerate(DE_list):
            pop.gen[:] = gEE_NMDA / gLeakE * (
                w_p * s_sel[k] + w_m * (np.sum(s_sel) - s_sel[k]) + w_m * s_dn
            )

        decisionEN.gen[:] = gEE_NMDA / gLeakE * s_all
        decisionI.gen[:] = gEI_NMDA / gLeakI * s_all

    C_IE = Synapses(decisionI, decisionE, 'w: 1',
                    on_pre='gi += w', delay=d_rec)
    C_II = Synapses(decisionI, decisionI, 'w: 1',
                    on_pre='gi += w', delay=d_rec)
    C_IE.connect()
    C_II.connect()
    C_IE.w = gIE_GABA / gLeakE
    C_II.w = gII_GABA / gLeakI

    # External Inputs
    ext_sel_groups, ext_sel_conns = [], []

    # Background currents from noises and other brain regions
    for k in range(num_classes):
        # A groups of Poisson Neurons with 3500 rates firing rates(Background current)
        pg = PoissonGroup(len(DE_list[k]), rates=nu_ext_sel)
        syn = Synapses(pg, DE_list[k], 'w : 1', on_pre='gea += w')
        syn.connect(j='i')
        syn.w = gextE / gLeakE
        ext_sel_groups.append(pg)
        ext_sel_conns.append(syn)

    extEN = PoissonGroup(len(decisionEN), rates=nu_ext_nsel)
    extI = PoissonGroup(NI, rates=nu_ext_nsel)

    synEN = Synapses(extEN, decisionEN, 'w : 1', on_pre='gea += w')
    synEN.connect(j='i')
    synEN.w = gextE / gLeakE

    synI = Synapses(extI, decisionI, 'w : 1', on_pre='gea += w')
    synI.connect(j='i')
    synI.w = gextI / gLeakI

    groups = {'DE': decisionE, 'DI': decisionI,
              'DX_sel': ext_sel_groups, 'DXN': extEN, 'DXI': extI}
    subgroups = {'DE_list': DE_list, 'DEN': decisionEN}
    conns = {
        'ampa_blocks': ampa_blocks,
        'selfnmda': selfnmda,
        'update_nmda': update_nmda,
        'C_EI_A': C_EI_A,
        'C_IE': C_IE,
        'C_II': C_II,
        'ext_sel_conns': ext_sel_conns,
        'synEN': synEN,
        'synI': synI
    }
    return groups, conns, subgroups

# ------------------------------------------------------------------------------
# 3. Trial Execution Function
# ------------------------------------------------------------------------------


def run_wang_on_triplet(
    triplet_item,
    N_images=1,
    win_thr_spikes=5,
    margin_thr=2,
    rt_bin_ms=5.0,
    record_monitors=True
):
    NUM_CLASSES = 2

    d_pos = triplet_item['dist_pos']
    d_neg = triplet_item['dist_neg']
    base_rate_hz = 15.0 
    scale = 20.0
    rate_pos = base_rate_hz + scale * d_pos
    rate_neg = base_rate_hz + scale * d_neg
    rates = [rate_pos, rate_neg]

    dt_s = float(dt_sim / second)
    stim_on_s = float(stim_on / second)
    stim_off_s = float(stim_off / second)

    steps_per_trial = int(np.round(trial_runtime / dt_sim))
    n_steps_total = steps_per_trial * N_images

    poisson_inputs_stim = []
    stim_synapses = []

    G, C, S = make_wang_circuit(num_classes=NUM_CLASSES)
    decisionE = G["DE"]
    decisionI = G["DI"]
    DE_list = S["DE_list"]

    for k in range(NUM_CLASSES):
        trace = np.zeros(n_steps_total, dtype=float)
        for img_idx in range(N_images):
            base = img_idx * steps_per_trial
            for step in range(steps_per_trial):
                t_in_trial = step * dt_s
                if (t_in_trial >= stim_on_s) and (t_in_trial < stim_off_s):
                    trace[base + step] = float(rates[k])  # Hz
                else:
                    trace[base + step] = 0.0

        current_ta = TimedArray(trace * Hz, dt=dt_sim)

        pg = PoissonGroup(len(DE_list[k]), rates='ta(t)', namespace={
                          'ta': current_ta})

        syn = Synapses(pg, DE_list[k], model="w:1",
                       on_pre="gea_post += w", delay=0.1 * ms)

        syn.connect(j="i")
        syn.w = w_ext_stim_val

        poisson_inputs_stim.append(pg)
        stim_synapses.append(syn)

    decisionE.run_regularly(
        """
        gea = 0
        gi = 0
        gen = 0
        xpre = 0
        spre = 0
        V = Vl + 2*mV*rand()
        """,
        dt=trial_runtime,
        when="start",
    )

    decisionI.run_regularly(
        """
        gea = 0
        gi = 0
        gen = 0
        V = Vl + 2*mV*rand()
        """,
        dt=trial_runtime,
        when="start",
    )

    spike_mons = [SpikeMonitor(DE_list[k]) for k in range(NUM_CLASSES)]
    rate_mons = [PopulationRateMonitor(DE_list[k]) for k in range(
        NUM_CLASSES)] if record_monitors else []
    if record_monitors:
        state_mon_d1 = StateMonitor(decisionE, 'V', record=0)
        state_mon_d2 = StateMonitor(decisionE, 'V', record=N_D)
    else:
        state_mon_d1 = None
        state_mon_d2 = None

    net = Network()
    net.add(decisionE, decisionI)

    for pg in G["DX_sel"]:
        net.add(pg)
    net.add(G["DXN"], G["DXI"])
    for syn in C["ext_sel_conns"]:
        net.add(syn)
    net.add(C["synEN"], C["synI"])

    for syn in C["ampa_blocks"]:
        net.add(syn)
    net.add(C["selfnmda"], C["update_nmda"])
    net.add(C["C_EI_A"], C["C_IE"], C["C_II"])

    for pg in poisson_inputs_stim:
        net.add(pg)
    for syn in stim_synapses:
        net.add(syn)

    for sm in spike_mons:
        net.add(sm)
    for rm in rate_mons:
        net.add(rm)
    if state_mon_d1 is not None:
        net.add(state_mon_d1)
    if state_mon_d2 is not None:
        net.add(state_mon_d2)

    decisionE.V = Vl
    decisionI.V = Vl
    decisionE.gea = 0
    decisionE.gi = 0
    decisionE.gen = 0
    decisionE.xpre = 0
    decisionE.spre = 0
    decisionI.gea = 0
    decisionI.gi = 0
    decisionI.gen = 0

    total_runtime = trial_runtime * N_images
    net.run(total_runtime)

    if record_monitors and state_mon_d1 is not None:
        print("--- StateMonitor Debug ---")
        print("V shape:", state_mon_d1.V.shape)

        if state_mon_d1.V.size > 0:
            v_vals = state_mon_d1.V[0] / mV
            print("Min V:", np.min(v_vals))
            print("Max V:", np.max(v_vals))
            print("Unique values in V:", np.unique(v_vals)[:5])
        else:
            print("WARNING: StateMonitor registered 0 time steps or 0 neurons!")

    counts = np.array([sm.num_spikes for sm in spike_mons])
    winner_idx = int(np.argmax(counts))
    class_names = ["Positive", "Negative"]

    sorted_counts = np.sort(counts)
    win = float(sorted_counts[-1])
    runner = float(sorted_counts[-2])
    conf = (win - runner) / (win + runner + 1e-6)

    rt_ms = np.nan
    rt_bin = rt_bin_ms * ms
    t_scan = stim_on_s
    spike_times = [np.asarray(sm.t / second, dtype=np.float64)
                   for sm in spike_mons]

    while t_scan < stim_off_s:
        cum = [int(np.sum(st < t_scan)) for st in spike_times]
        cum_sorted = np.sort(cum)
        w_c = int(cum_sorted[-1])
        r_c = int(cum_sorted[-2])

        if (w_c >= win_thr_spikes) and ((w_c - r_c) >= margin_thr):
            rt_ms = float((t_scan - stim_on_s) * 1000.0)
            break
        t_scan += float(rt_bin / second)

    res = {
        'triplet_id': triplet_item['triplet_id'],
        'anchor': triplet_item['anchor'],
        'positive': triplet_item['positive'],
        'negative': triplet_item['negative'],
        'dist_pos': triplet_item['dist_pos'],
        'dist_neg': triplet_item['dist_neg'],
        'rate_pos_hz': rate_pos,
        'rate_neg_hz': rate_neg,
        'decision': class_names[winner_idx],
        'decision_idx': winner_idx,
        'spike_counts': counts.tolist(),
        'confidence': float(conf),
        'rt_ms': rt_ms
    }

    if record_monitors:
        res['spike_mons'] = spike_mons
        res['rate_mons'] = rate_mons
        res['state_mon_d1'] = state_mon_d1
        res['state_mon_d2'] = state_mon_d2

    return res

# ------------------------------------------------------------------------------
# 4. Diagnostics & Visualization
# ------------------------------------------------------------------------------


def plot_wang_results(result):
    if 'spike_mons' not in result or result['spike_mons'] is None:
        print(
            "Set 'record_monitors=True' in run_wang_on_triplet to display diagnostic plots.")
        return

    spike_mons = result['spike_mons']
    rate_mons = result['rate_mons']
    sm1 = result.get('state_mon_d1')
    sm2 = result.get('state_mon_d2')

    fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)

    colors = ['#1f77b4', '#ff7f0e']
    labels = ['Positive Pool (D1)', 'Negative Pool (D2)']

    # 1. Firing Rates
    ax = axes[0]
    for k in range(len(rate_mons)):
        smooth_r = rate_mons[k].smooth_rate(window='gaussian', width=20 * ms)
        ax.plot(rate_mons[k].t / ms, smooth_r / Hz,
                color=colors[k], label=labels[k], lw=2)
    ax.axvspan(stim_on / ms, stim_off / ms, color='gray',
               alpha=0.15, label='Stimulus Window')
    ax.set_ylabel("Rate (Hz)")
    ax.set_title("Population Firing Rates")
    ax.legend(loc="upper right")
    ax.grid(True, linestyle=":", alpha=0.6)

    # 2. Spike Raster
    ax = axes[1]
    for k in range(len(spike_mons)):
        ax.plot(spike_mons[k].t / ms, spike_mons[k].i + k *
                N_D, '.', color=colors[k], ms=2, label=labels[k])
    ax.axvspan(stim_on / ms, stim_off / ms, color='gray', alpha=0.15)
    ax.set_ylabel("Neuron Index")
    ax.set_title("Spike Raster Plot")
    ax.legend(loc="upper right")
    ax.grid(True, linestyle=":", alpha=0.6)

    # 3. Voltage Dynamics
    ax = axes[2]
    if sm1 is not None and sm2 is not None and sm1.V.size > 0:
        ax.plot(sm1.t / ms, sm1.V[0] / mV,
                label="Sample Neuron D1 (Pos)", color=colors[0], alpha=0.85)
        ax.plot(sm2.t / ms, sm2.V[0] / mV,
                label="Sample Neuron D2 (Neg)", color=colors[1], alpha=0.85)
        ax.axhline(Vt / mV, color='red', linestyle='--',
                   alpha=0.7, label='Threshold (-60 mV)')
        spk_times_d2 = spike_mons[1].t[spike_mons[1].i == 0] / ms
        if len(spk_times_d2) > 0:
            ax.plot(spk_times_d2, np.ones_like(spk_times_d2) * (Vt / mV),
                    '|', color='black', ms=10, label='Spikes (Neuron 0)')

    plt.tight_layout()
    plt.show()


In [ ]:
def visualize_lora_quality(dataset):
    d_pos = np.array([item['dist_pos'] for item in dataset])
    d_neg = np.array([item['dist_neg'] for item in dataset])
    margins = d_neg - d_pos

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].hist(d_pos, bins=25, color='#2a9d8f', alpha=0.6,
                 label='D_pos (Anchor-Pos)', edgecolor='k')
    axes[0].hist(d_neg, bins=25, color='#e76f51', alpha=0.6,
                 label='D_neg (Anchor-Neg)', edgecolor='k')
    axes[0].set_title('Embedding Distance Distributions',
                      fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Distance Value')
    axes[0].set_ylabel('Triplet Count')
    axes[0].legend()
    axes[0].grid(True, linestyle='--', alpha=0.5)

    correct_mask = margins > 0
    axes[1].hist(margins[correct_mask], bins=20, color='#2b5c8f',
                 alpha=0.7, edgecolor='k', label='Correct Separation (M > 0)')
    axes[1].hist(margins[~correct_mask], bins=20, color='#d62828',
                 alpha=0.7, edgecolor='k', label='Wrong Separation (M <= 0)')
    axes[1].axvline(0, color='black', linestyle='--', linewidth=2)
    axes[1].set_title(
        'Separation Margin (D_{neg} - D_{pos})', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Margin Difference')
    axes[1].set_ylabel('Triplet Count')
    axes[1].legend()
    axes[1].grid(True, linestyle='--', alpha=0.5)

    axes[2].scatter(d_pos, d_neg, c=margins, cmap='coolwarm',
                    alpha=0.8, edgecolors='k')
    max_val = max(d_pos.max(), d_neg.max())
    axes[2].plot([0, max_val], [0, max_val], 'r--',
                 label='Equal Distance Line (D_{pos}=D_{neg})')
    axes[2].set_title(
        'Scatter: D_{pos} vs D_{neg}', fontsize=12, fontweight='bold')
    axes[2].set_xlabel('D_{pos}$')
    axes[2].set_ylabel('D_{neg}$')
    axes[2].legend()
    axes[2].grid(True, linestyle='--', alpha=0.5)

    plt.tight_layout()
    plt.show()

    acc = (margins > 0).mean() * 100
    print("===================================")
    print("   LoRA Embedding Quality Report   ")
    print("===================================")
    print(f"• Correct Triplet Ordering (D_neg > D_pos) : {acc:.2f}%")
    print(f"• Mean Margin (D_neg - D_pos)            : {margins.mean():.4f}")
    print(f"• Margin Std Dev                         : {margins.std():.4f}")
    print(
        f"• Min / Max Margin                       : {margins.min():.4f} / {margins.max():.4f}")


def visualize_wang_input_drive(dataset, base_rate=15.0, scale=20.0):
    r_pos = np.array([base_rate + scale * item['dist_pos']
                     for item in dataset])
    r_neg = np.array([base_rate + scale * item['dist_neg']
                     for item in dataset])
    delta_rates = np.abs(r_neg - r_pos)

    plt.figure(figsize=(9, 4.5))
    plt.hist(delta_rates, bins=25, color='#457b9d', edgecolor='k', alpha=0.7)
    plt.axvline(1.0, color='orange', linestyle='--',
                linewidth=2, label='Weak Competence (~1 Hz)')
    plt.axvline(5.0, color='green', linestyle='--', linewidth=2,
                label='Strong Attractor Dynamics (~5 Hz)')

    plt.title('Predicted Input Firing Rate Difference (Delta Rate$ for Wang Circuit)',
              fontsize=12, fontweight='bold')
    plt.xlabel('Delta Rate = |r_{neg} - r_{pos}|(Hz)')
    plt.ylabel('Triplet Count')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()


with open(json_path, 'r', encoding='utf-8') as f:
    train_data = json.load(f)

visualize_wang_input_drive(train_data)

visualize_lora_quality(train_data)

In [ ]:
def evaluate_and_plot_psychometrics(dataset, title_prefix="Tuned Model"):
    delta_rates = []
    rts = []
    confidences = []

    print("⏳ Running evaluation on dataset for psychometric analysis...")
    for item in dataset:
        res = run_wang_on_triplet(item, record_monitors=False)
        rt = res['rt_ms']

        if not np.isnan(rt):

            delta_r = abs(res['rate_neg_hz'] - res['rate_pos_hz'])
            delta_rates.append(delta_r)
            rts.append(rt)
            confidences.append(res['confidence'])

    delta_rates = np.array(delta_rates)
    rts = np.array(rts)
    confidences = np.array(confidences)

    if len(rts) < 2:
        print("❌ Not enough valid decisions to fit regression lines!")
        return

    slope_rt, intercept_rt = np.polyfit(delta_rates, rts, 1)
    r_rt = np.corrcoef(delta_rates, rts)[0, 1]

    slope_conf, intercept_conf = np.polyfit(rts, confidences, 1)
    r_conf = np.corrcoef(rts, confidences)[0, 1]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

    axes[0].scatter(delta_rates, rts, color='#2b5c8f',
                    alpha=0.7, edgecolors='k', label='Triplets')
    x_fit_rt = np.linspace(delta_rates.min(), delta_rates.max(), 100)
    axes[0].plot(x_fit_rt, slope_rt * x_fit_rt + intercept_rt, color='#e63946', linewidth=2.5,
                 label=f'Fit: Slope={slope_rt:.2f}\nR={r_rt:.2f}')

    axes[0].set_title(
        f'{title_prefix}: Reaction Time vs Input Differences', fontsize=12, fontweight='bold')
    axes[0].set_xlabel(
        r'$\Delta$ Rate ($|r_{neg} - r_{pos}|$) [Hz]', fontsize=11)
    axes[0].set_ylabel('Reaction Time (ms)', fontsize=11)
    axes[0].grid(True, linestyle='--', alpha=0.5)
    axes[0].legend(loc='upper right', frameon=True)

    axes[1].scatter(rts, confidences, color='#2a9d8f',
                    alpha=0.7, edgecolors='k', label='Triplets')
    x_fit_conf = np.linspace(rts.min(), rts.max(), 100)
    axes[1].plot(x_fit_conf, slope_conf * x_fit_conf + intercept_conf, color='#d90429', linewidth=2.5,
                 label=f'Fit: Slope={slope_conf:.4f}\nR={r_conf:.2f}')

    axes[1].set_title(
        f'{title_prefix}: Confidence vs Reaction Time', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Reaction Time (ms)', fontsize=11)
    axes[1].set_ylabel('Confidence', fontsize=11)
    axes[1].grid(True, linestyle='--', alpha=0.5)
    axes[1].legend(loc='upper right', frameon=True)

    plt.tight_layout()
    plt.show()

    print("\n===================================")
    print("   Psychometric Trend Analysis")
    print("===================================")
    print(
        f"• RT vs Delta Rate  -> Slope: {slope_rt:.3f} | Pearson R: {r_rt:.3f}")
    print(
        f"• Conf vs RT        -> Slope: {slope_conf:.5f} | Pearson R: {r_conf:.3f}")

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)
brian2.BrianLogger.suppress_name('unused_brian_object')
try:
    device.reinit()
except Exception:
    pass
set_device('runtime')
prefs.codegen.target = 'cython'
start_scope()


def objective_psychometric(trial, dataset_sample):
    global w_p, w_m
    global nu_ext_sel, nu_ext_nsel, w_ext_stim_val
    global win_thr_spikes, margin_thr, rt_bin_ms

    w_p_val = trial.suggest_float('w_p', 1.3, 1.9)
    w_p = w_p_val
    w_m = (1.0 - f_E * (w_p - 1.0) / (1.0 - f_E)) * 0.9

    nu_ext_sel_val = trial.suggest_float('nu_ext_sel_hz', 2350.0, 2500.0)
    nu_ext_nsel_val = trial.suggest_float('nu_ext_nsel_hz', 2350.0, 2450.0)
    w_ext_stim_val_opt = trial.suggest_float('w_ext_stim_val', 0.10, 0.22)

    nu_ext_sel = nu_ext_sel_val * Hz
    nu_ext_nsel = nu_ext_nsel_val * Hz
    w_ext_stim_val = w_ext_stim_val_opt

    win_thr_spikes = trial.suggest_int('win_thr_spikes', 20, 80)
    margin_thr = trial.suggest_int('margin_thr', 10, 40)
    rt_bin_ms = trial.suggest_float('rt_bin_ms', 5.0, 20.0)

    delta_rates = []
    rts = []
    confidences = []
    penalties = 0.0

    for item in dataset_sample:
        try:
            res = run_wang_on_triplet(item, record_monitors=False)

            rt = res['rt_ms']
            conf = res['confidence']
            delta_r = abs(res['rate_pos_hz'] - res['rate_neg_hz'])

            if res.get('decision_idx', -1) != 1:
                penalties += 100.0

            if np.isnan(rt):
                penalties += 200.0
            else:
                delta_rates.append(delta_r)
                rts.append(rt)
                confidences.append(conf)

                if rt < 250.0 or rt > 1500.0:
                    penalties += 50.0

        except Exception:
            penalties += 500.0

    if len(rts) < (len(dataset_sample) * 0.5):
        return 9999.0

    delta_rates = np.array(delta_rates)
    rts = np.array(rts)
    confidences = np.array(confidences)

    if np.std(delta_rates) > 0 and np.std(rts) > 0:
        corr_delta_rt = np.corrcoef(delta_rates, rts)[0, 1]
    else:
        corr_delta_rt = 0.0

    if np.std(rts) > 0 and np.std(confidences) > 0:
        corr_rt_conf = np.corrcoef(rts, confidences)[0, 1]
    else:
        corr_rt_conf = 0.0

    psychometric_loss = 0.0

    if corr_delta_rt > -0.2:
        psychometric_loss += (corr_delta_rt + 1.0) * 200.0

    if corr_rt_conf > -0.2:
        psychometric_loss += (corr_rt_conf + 1.0) * 200.0

    return penalties + psychometric_loss


print("📂 Loading dataset for Wang parameter tuning...")

with open(json_path, "r", encoding="utf-8") as f:
    train_data = json.load(f)

optuna_sample = train_data[:5]

print(
    f"🚀 Starting Optuna Search on Specific Parameters ({len(optuna_sample)} triplets)...")
study = optuna.create_study(direction="minimize")

study.optimize(lambda trial: objective_psychometric(trial, optuna_sample),
               n_trials=12, show_progress_bar=True)

print("\n🎉 Optimization Complete!")
print(" Optimal Values Found:")
for k, v in study.best_params.items():
    print(f"   • {k}: {v:.4f}" if isinstance(v, float) else f"   • {k}: {v}")

w_p = study.best_params['w_p']
w_m = (1.0 - f_E * (w_p - 1.0) / (1.0 - f_E)) * 0.9

nu_ext_sel = study.best_params['nu_ext_sel_hz'] * Hz
nu_ext_nsel = study.best_params['nu_ext_nsel_hz'] * Hz
w_ext_stim_val = study.best_params['w_ext_stim_val']

win_thr_spikes = study.best_params['win_thr_spikes']
margin_thr = study.best_params['margin_thr']
rt_bin_ms = study.best_params['rt_bin_ms']

print("\n🔮 Testing optimal parameters on sample triplet with full monitors...")
test_result = run_wang_on_triplet(train_data[0], record_monitors=True)

print("\n===================================")
print("   Tuned Wang Circuit Result")
print("===================================")
print(f"Decision      : {test_result['decision']}")
print(
    f"Spike Counts  : Pos={test_result['spike_counts'][0]} | Neg={test_result['spike_counts'][1]}")
print(f"Confidence    : {test_result['confidence']:.3f}")
print(f"Reaction Time : {test_result['rt_ms']} ms" if not np.isnan(
    test_result['rt_ms']) else "Reaction Time : No Decision")

plot_wang_results(test_result)
evaluate_and_plot_psychometrics(
    train_data[:20], title_prefix='Tuned Wang Model'
)